# 5. Andmete eeltöötlus


Sisukord:
* [Puuduvad andmed](#puudu)
* [Klasside tasakaalustamine](#tasakaal)
* [Ülesanne 5.1](#5_1)
* [Sobimatut tüüpi (nominaal-, ordinaal-) andmete konverteerimine](#sobimatu)
* [Andmete skaleerimine: normaliseerimine ja standardiseerimine](#skaleeri)
* [Oluliste atribuutide väljavalimine](#atr)
* [Ülesanne 5.2](#5_2)

Vaata ka
* [SciKit-learn.preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing)

Vt ka: Sebastian Raschka. Python Machine Learning. Ch. 4. Building Good Training Sets: Data Preprocessing.

## Ebaõiged andmed

Olemasolevad, aga ebaõiged andmed on probleem, mille avastamine ja parandamine on küllaltki keerukas. Andmekvaliteedi hindamine võib eeldada eksperthinnangut piisavalt suurele valimile andmetest. Seejärel on võimalik vigade parandamiseks treenida ja rakendada ennustavaid mudeleid.

<a id='puudu'></a>
## Puuduvad andmed

Reaalsete andmestike korral on sagedaseks probleemiks puuduvad andmed: küsitletav ei vasta mõnele küsimusele, ajaloolised andmed on puudulikud jne. Enamik andmekaeve meetodeid jäävad sellise sisendiga hätta. Tüüpilisteks viisideks puuduvate andmetega ümberkäimisel on vastavate ridade/veergude väljaviskamine või puuduvate väärtuste asendamine keskväärtusega.

In [1]:
import pandas as pd
from io import StringIO # StringIO klass esitab stringi faililaadse objektina

# Tekitame puuduvate väärtustega NaN (Not a Number) andmeraamistiku df, kasutades read_csv() meetodit
csv_data = """A,B,C,D
1.0, 2.0, 3.0, 4.0
5.0, 6.0,, 8.0
0.0, 11.0, 12.0,"""
df = pd.read_csv(StringIO(csv_data))
df

C:\Users\Maive\AppData\Local\Temp\ipykernel_28236\3492276596.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,0.0,11.0,12.0,NaN


### Väljaviskamine

Kasutame `pandas DataFrame` meetodit [dropna()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html).

In [2]:
print("Puuduvad väärtused veeruti\n", df.isnull().sum(axis=0))

Puuduvad väärtused veeruti
 A    0
B    0
C    1
D    1
dtype: int64


In [3]:
# Elimineerime puuduvate väärtustega read
df_clean1 = df.dropna()
df_clean1

,A,B,C,D
0,1.0,2.0,3.0,4.0


In [4]:
 #Elimineerime puuduvate väärtustega veerud
df_clean2 = df.dropna(axis=1)
df_clean2

,A,B
0,1.0,2.0
1,5.0,6.0
2,0.0,11.0


In [5]:
# Viska välja ainult need read, kus kõik väärtused puuduvad
df.dropna(how="all")

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,0.0,11.0,12.0,NaN


In [6]:
# Jäta alles need read, kus on vähemalt 4 väärtust
df.dropna(thresh=4, axis=0)

,A,B,C,D
0,1.0,2.0,3.0,4.0


In [7]:
# Viska välja ainult need read, kus puuduvad veergude B või C väärtused
df.dropna(subset=['B', 'C'])

,A,B,C,D
0,1.0,2.0,3.0,4.0
2,0.0,11.0,12.0,NaN


### Asendamine

In [8]:
# Kasutame klassi Imputer, et asendada puuduvad väärtused (missing_values="NaN") keskmistega (strategy="mean")
#from sklearn.preprocessing import Imputer
from sklearn.impute import SimpleImputer
import numpy as np

imr = SimpleImputer(missing_values=np.nan, strategy="mean") 
imputed_data = imr.fit_transform(df)
imputed_data

array([[ 1. ,  2. ,  3. ,  4. ],
       [ 5. ,  6. ,  7.5,  8. ],
       [ 0. , 11. , 12. ,  6. ]])

In [9]:
# Asendatud imputed_data on np.array. 
# Teeme sellest pandas DataFrame objekti, millel on algsega samad veerunimed
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,7.5,8.0
2,0.0,11.0,12.0,6.0


Siin võib kasutada ka pandas meetodit [fillna()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html#pandas.DataFrame.fillna), millele antud näites anname asenduseks ette veergude keskmiste vektori (`df.mean()`). Tulemus on eelmisega samaväärne.

In [10]:
df.mean()

A    2.000000
B    6.333333
C    7.500000
D    6.000000
dtype: float64

In [11]:
fill_df = df.fillna(df.mean())

In [12]:
fill_df

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,7.5,8.0
2,0.0,11.0,12.0,6.0


Võimsama meetodina võib mainida puuduvate väärtuste imputeerimist $k$ lähinaabri abil ([kNN](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html)), mis arvutab puuduva väärtuse k kõige lähema objekti keskmisena. Tüüpiliseks kaugusmõõduks ehk sarnasusmõõduks on siin Eukleidese kaugus.

In [13]:
from sklearn.impute import KNNImputer

knn_imp = KNNImputer(missing_values=np.nan, n_neighbors=1) # k=1 seoses näite pisikese tabeliga, vaikimisi 5
knn_imputed_data = knn_imp.fit_transform(df)
knn_imputed_data

array([[ 1.,  2.,  3.,  4.],
       [ 5.,  6.,  3.,  8.],
       [ 0., 11., 12.,  8.]])

<a id='tasakaal'></a>
## Klasside tasakaalustamine

Kui mõnda klassi esineb andmetes vähe, siis ennustavad mudelid ei pruugi seda edukalt tuvastada. Sellisel juhul on meil tegemist tasakaalustamata klassidega. Meetodid selle probleemiga tegelemiseks on: 
* Juhuslik ülevalimine, kus haruldastesse klassidesse kuuluvaid objekte juhuslikult kopeeritakse (näiteks SMOTE meetodil).
* Juhuslik alavalimine, kus levinud klassidesse kuuluvaid objekte juhuslikult kustutatakse.
* Sünteetiliste andmete genereerimine, kus genereeritakse olemasolevate haruldaste objektidega sarnaseid objekte (näiteks GAN meetodil).
* Ennustavas mudelis klassidele erinevate kaalude andmine. Sklearn paketis teeb seda hüperparameeter *class_weights*.



<a id='5_1'></a>
## Ülesanne 5.1

Eemaldada allolevast andmeraamistikust `df_ex` veerud, kus puuduvad kõik väärtused. Asendada ülejäänud  puuduvad väärtused keskmistega üle veergude.

<!-- Võtta aluseks [UCI Horse Colic (hobuste kõhuvalu) andmestik](https://archive.ics.uci.edu/ml/datasets/Horse+Colic), mille saab alla laadida https://archive.ics.uci.edu/ml/machine-learning-databases/horse-colic/horse-colic.data. -->

In [14]:
csv_ex = """A,B,C,D,E
1.0, 2.0, 3.0, 24.0,
15.0, 6.0,,8.0,
0.0, 11.0, 9.0,,"""
df_ex = pd.read_csv(StringIO(csv_ex))
df_ex

,A,B,C,D,E
0,1.0,2.0,3.0,24.0,NaN
1,15.0,6.0,NaN,8.0,NaN
2,0.0,11.0,9.0,NaN,NaN


<!--
"""
cols =                                  ["surgery?",
                                         "Age",
                                         "Hospital Nr",
                                         "rectal temp",
                                         "pulse",
                                         "respiratory rate",
                                         "temp of extermities",
                                         "peripheral pulse",
                                         "mucuos membranes",
                                         "capillary refill time",
                                         "pain",
                                         "peristalsis",
                                         "abdominal distension",
                                         "nasogastric tube"
                                         "nasogastric reflux",
                                         "nasogastric reflux PH",
                                         "rectal examination-feces",
                                         "abdomen",
                                         "packed cell volume",
                                         "total protein",
                                         "abdominocentesis appearance",
                                         "abdominocentesis total protein",
                                         "outcome",
                                         "surgical lesion?",
                                         "type of lesion 1",
                                         "type of lesion 2",
                                         "type of lesion 3",
                                         "cp_data"
                                         ]
df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/horse-colic/horse-colic.data", 
                 delimiter=" ",  index_col=False)
#print(list(zip(df.columns, cols)))
#print(df.columns)
#print(cols)
df
"""
-->

In [15]:
df_ex = df_ex.dropna(axis=1, how='all')
df_ex

,A,B,C,D
0,1.0,2.0,3.0,24.0
1,15.0,6.0,NaN,8.0
2,0.0,11.0,9.0,NaN


In [16]:
# Kasutame klassi Imputer, et asendada puuduvad väärtused (missing_values="NaN") keskmistega (strategy="mean")
imr = SimpleImputer(missing_values=np.nan, strategy="mean") 
imputed_data = imr.fit_transform(df_ex)
imputed_df = pd.DataFrame(imputed_data, columns=df_ex.columns)
imputed_df 

,A,B,C,D
0,1.0,2.0,3.0,24.0
1,15.0,6.0,6.0,8.0
2,0.0,11.0,9.0,16.0


<a id='sobimatu'></a>
## Sobimatut tüüpi (nominaal-, ordinaal-) andmete konverteerimine 

Lisaks reaal- ja täisarvudele võivad andmed olla ka nominaal- või ordinaalandmed,

* Nominaalandmed: kategooria, millel puudub järjestus. Näiteks: veregrupp A, B, AB, O
* Ordinaalandmed: järjestus on, aga astmete vahe ei pruugi olla üle skaala ühtne. N: meeldib väga, meeldib, neutraalne, ei meeldi, ei meeldi üldse. 

Mõned meetodid (näiteks pertseptron) ootavad arvandmeid ja nende jaoks võib olla vaja rohkem kui kahe väärtusega nominaal- ja vahel ka ordinaalatribuute teisendada. Kahe väärtusega atribuudi võib teisendada kahendkujule (0, 1).

In [17]:
patsiendi_df = pd.DataFrame({"veregrupp": ["A", "AB", "A", "O", "O"], 
                             "valu": ["puudub", "puudub", "kerge", "intensiivne", "kerge"]})
patsiendi_df                            

,veregrupp,valu
0,A,puudub
1,AB,puudub
2,A,kerge
3,O,intensiivne
4,O,kerge


Ordinaalatribuutide puhul piisab tihti siltide teisendamisest täisarvulisele skaalale. Siin sobib hästi kasutada nö. *mapping* sõnastikke, mille võti on väärtus vanal skaalal ja väärtus on väärtus uuel skaalal. `Dataframe[col].map(mapping)` meetod teisendab seejärel vastava veeru.

In [18]:
valu_map = {"puudub": 0, "kerge": 1, "intensiivne": 2}
patsiendi_df["valu"] = patsiendi_df["valu"].map(valu_map)
patsiendi_df

,veregrupp,valu
0,A,0
1,AB,0
2,A,1
3,O,2
4,O,1


Nominaalatribuutide puhul on selline lähenemine ebasobiv. Arvuline järjestus oleks petlik, sest sisuline järjestus puudub. Siin tekitatakse iga nominaalkategooria jaoks oma 0-1 atribuut (0-polnud see väärtus, 1-oli see väärtus). Seda võimaldab `sklearn` alammooduli `preprocessing`klass [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), mis töötab arvandmete peal  või `pandas`mooduli funktsioon  [get_dummies(df)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html), mis teisendab stringe sisaldavad veerud.

In [19]:
patsiendi_df = pd.get_dummies(patsiendi_df)
patsiendi_df

,valu,veregrupp_A,veregrupp_AB,veregrupp_O
0,0,True,False,False
1,0,False,True,False
2,1,True,False,False
3,2,False,False,True
4,1,False,False,True


Kui meil on vaja rakendada erinevatele veergudele erinevaid teisenduspoliitikaid, siis võib kasutada klassi [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html#sklearn.compose.ColumnTransformer), mis seab igale veerule vastavusse transformaatorobjekti (peab omama `fit()` ja `transform()` meetodeid. ColumnTransformeri initsialiseerimise sisendiks on (nime_str, transformaator, veeru_id või veeru_id list) kolmikute list. 

In [20]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

patsiendi_df2 = pd.DataFrame({"veregrupp": ["A", "AB", "A", "O", "O"], 
                             "valu": ["puudub", "puudub", "kerge", "intensiivne", "kerge"]})

# NB! OrdinalEncoder järjestab stringid tähestikulises järjekorras, sisuliselt nominaalskaala, mitte ordinaalskaala
ct = ColumnTransformer([("int_valu", OrdinalEncoder(categories=[["intensiivne", "kerge", "puudub"]]), [1]),("binarize_veri", OneHotEncoder(), [0])])
ct.fit_transform(patsiendi_df2)

array([[2., 1., 0., 0.],
       [2., 0., 1., 0.],
       [1., 1., 0., 0.],
       [0., 0., 0., 1.],
       [1., 0., 0., 1.]])

<a id='skaleeri'></a>
## Andmete skaleerimine: normaliseerimine ja standardiseerimine

Enamik masinõppe algoritme, va otsustuspuud, töötavad paremini kui kõik atribuudid on samal skaalal.
On kaks fundamentaalset lähenemist:
* Viimine  \[0..1\] skaalale (Vahel nimetatud ka **Normaliseerimiseks**).
* **Standardiseerimine** skaalale, kus keskväärtus on 0 ja standardhälve on 1.

Normaliseeritava veeru $x$  ja rea $i$ elemendi uus väärtus $x_{norm}^i$:
$$ x_{norm}^i = \frac{x^i - x_{min}}{x_{max} - x_{min}}$$
$x^i$: vana väärtus; $x_{max}, x_{min}$: veeru $x$ maksimaalne ja minimaalne väärtus.
Normaliseerimise eest vastutav klass on [sklearn.preprocessing.MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html).

Standardiseeritava veeru $x$  ja rea $i$ elemendi uus väärtus $x_{std}^i$:
$$ x_{std}^i = \frac{x^i - \mu_x}{\sigma_x}$$
$x^i$: vana väärtus; $\mu_x, \sigma_x$: veeru $x$ keskväärtus ja standardhälve.
Standardiseerimise eest vastutav klass on [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).


In [21]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
norm_data = MinMaxScaler().fit_transform(patsiendi_df)
print(norm_data)
# Tekitame uue normaliseeritud dataframe objekti
patsiendi_df_norm = pd.DataFrame(norm_data, columns=patsiendi_df.columns)
display(patsiendi_df_norm)

[[0.  1.  0.  0. ]
 [0.  0.  1.  0. ]
 [0.5 1.  0.  0. ]
 [1.  0.  0.  1. ]
 [0.5 0.  0.  1. ]]


,valu,veregrupp_A,veregrupp_AB,veregrupp_O
0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0
2,0.5,1.0,0.0,0.0
3,1.0,0.0,0.0,1.0
4,0.5,0.0,0.0,1.0


In [22]:
# Teine võimalus on muuta olemasolevat Dataframe objekti 
# loc atribuudi ja maksimaalse lõike abil.
patsiendi_df.loc[:,:] = StandardScaler().fit_transform(patsiendi_df)
patsiendi_df

C:\Users\Maive\AppData\Local\Temp\ipykernel_28236\1434368823.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.06904497 -1.06904497  0.26726124  1.60356745  0.26726124]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  patsiendi_df.loc[:,:] = StandardScaler().fit_transform(patsiendi_df)
C:\Users\Maive\AppData\Local\Temp\ipykernel_28236\1434368823.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.22474487 -0.81649658  1.22474487 -0.81649658 -0.81649658]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  patsiendi_df.loc[:,:] = StandardScaler().fit_transform(patsiendi_df)
C:\Users\Maive\AppData\Local\Temp\ipykernel_28236\1434368823.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.

,valu,veregrupp_A,veregrupp_AB,veregrupp_O
0,-1.069045,1.224745,-0.5,-0.816497
1,-1.069045,-0.816497,2.0,-0.816497
2,0.267261,1.224745,-0.5,-0.816497
3,1.603567,-0.816497,-0.5,1.224745
4,0.267261,-0.816497,-0.5,1.224745


In [23]:
patsiendi_df.describe()

,valu,veregrupp_A,veregrupp_AB,veregrupp_O
count,5.000000e+00,5.000000e+00,5.000000e+00,5.000000
mean,-8.881784e-17,-4.440892e-17,-2.220446e-17,0.000000
std,1.118034e+00,1.118034e+00,1.118034e+00,1.118034
min,-1.069045e+00,-8.164966e-01,-5.000000e-01,-0.816497
25%,-1.069045e+00,-8.164966e-01,-5.000000e-01,-0.816497
50%,2.672612e-01,-8.164966e-01,-5.000000e-01,-0.816497
75%,2.672612e-01,1.224745e+00,-5.000000e-01,1.224745
max,1.603567e+00,1.224745e+00,2.000000e+00,1.224745


<a id='atr'></a>
## Oluliste atribuutide väljavalimine (*feature selection*)

Suure arvu atribuutide korral, eriti kui objektide arv on suhteliselt väike, on ülekohandamise (overfitting) probleem lihtne tekkima: objekti klassi saab ennustada atribuutide kombinatsiooni alusel, mis on unikaalne üksikobjektile. Selliselt treenitud klassifikaatorid annavad näiliselt häid tulemusi treeningandmetel, aga ennustustäpsus langeb tugevalt uute andmete korral. Samuti on sellised mudelid keerulised ja väikese üldistusjõuga. 

Seega on tihti kasulik atribuutide arvu vähendada. Siin on kaks fundamentaalset strateegiat:
* Uute atribuutide defineerimine olemasolevate atribuutide kombinatsioonina, nö dimensionaalsuse vähendamine/leidmine, mida vaatame järgmise nädala teema all. (*feature extraction*)
* Olemasolevatest atribuutidest sobivaimate väljavalimine. (*feature selection*)



Tihti on võimalik vähendada ennustamiseks kasutatavate atribuutide arvu klassifikaatorispetsiifiliste meetodite abil. Näiteks logistilise regressiooni korral suurendab nullkaalude arvu `penalty="l1"` (L1 regulariseerimine ehk lasso) koos madala hinnaargumendiga `C`.

https://scikit-learn.org/stable/auto_examples/linear_model/plot_logistic_l1_l2_sparsity.html

Kaaluvektori **w** hinnafunktsioonile J(**w**) lisandub regulariseerimise trahv L1, L2 või muu:

L1: $$ \sum_{j=1..m} |w_j|$$

L2 (väiksem trahv väikestele $w_j$ väärtustele): $$ \sum_{j=1..m} w_j^2$$

Kui $|w_j| < 1$, siis on L1 trahv suurem kui L2 oma, muidu vastupidi. Seega on L1 trahvi korral suurem tendents ebaoluliste atribuutide kaalud üldse nullida. L2 trahv viib need kaalud väikeseks, aga mitte nulliks.


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn import datasets

iris = datasets.load_iris()
clf_general = LogisticRegression(penalty="l2", C=0.1, solver='liblinear')
clf_general.fit(iris.data, iris.target)
print("Tavalise logistilise regressiooni (penalty='l2' ja C=0.1) leitud kaalud:")
print(clf_general.coef_)

clf_sparse = LogisticRegression(penalty="l1", C=0.1, solver='liblinear')
clf_sparse.fit(iris.data, iris.target)
print("\n\nLogistilise regressiooni leitud kaalud, kui penalty='l1' ja C=0.1:")
print(clf_sparse.coef_)


Tavalise logistilise regressiooni (penalty='l2' ja C=0.1) leitud kaalud:
[[ 0.21310863  0.776912   -1.23987617 -0.55518357]
 [ 0.04423007 -0.62688956  0.29231476 -0.24855524]
 [-0.63387401 -0.5619517   0.94991857  0.78619837]]


Logistilise regressiooni leitud kaalud, kui penalty='l1' ja C=0.1:
[[ 0.          1.12162618 -1.34352873  0.        ]
 [ 0.         -0.38708389  0.12332826  0.        ]
 [-0.98734752  0.          1.27637182  0.        ]]


In [25]:
# Vormistame selle ilusa ja loetava DataFrame'ina
print("Tava:")
general_coef_df = pd.DataFrame(clf_general.coef_, 
                             columns=iris.feature_names,
                             index=iris.target_names)
general_coef_df

Tava:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
setosa,0.213109,0.776912,-1.239876,-0.555184
versicolor,0.044230,-0.626890,0.292315,-0.248555
virginica,-0.633874,-0.561952,0.949919,0.786198


In [26]:
# Vormistame selle ilusa ja loetava DataFrame'ina
print("penalty='l1' ja C=0.1:")
sparse_coef_df = pd.DataFrame(clf_sparse.coef_, 
                             columns=iris.feature_names,
                             index=iris.target_names)
sparse_coef_df

penalty='l1' ja C=0.1:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
setosa,0.000000,1.121626,-1.343529,0.0
versicolor,0.000000,-0.387084,0.123328,0.0
virginica,-0.987348,0.000000,1.276372,0.0


In [27]:
# Siia võiks tulla ka täpsuse (või muu headuse mõõdu) analüüs

Klass [SelectKBest](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html) võimaldab valida välja $k$ parimat atribuuti teatud hinnangufunktsiooni järgi. Hinnangufunktsioon võib olla statistiline ($\chi^2$, ANOVA F statistik,...) või näiteks valepositiivsete tulemuste arv atribuudi järgi ennustades.

In [28]:
from sklearn.feature_selection import SelectKBest, chi2

# Tekitame SelectKBest transformaatori,
# treenime ja saame vastuse fit_transform() meetodi abil
k_best = SelectKBest(chi2, k=2)
X_new = k_best.fit_transform(iris.data, iris.target)
print(k_best.scores_)
X_new[:10]

[ 10.81782088   3.7107283  116.31261309  67.0483602 ]


array([[1.4, 0.2],
       [1.4, 0.2],
       [1.3, 0.2],
       [1.5, 0.2],
       [1.4, 0.2],
       [1.7, 0.4],
       [1.4, 0.3],
       [1.5, 0.2],
       [1.4, 0.2],
       [1.5, 0.1]])

Atribuutide väljavalimiseks saame kasutada ka juhusliku metsa [RandomForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) ansamblimeetodi omadust `feature_importances_`.

In [29]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=-1)
forest.fit(iris.data, iris.target)
print(forest.feature_importances_)

[0.09792816 0.02518223 0.4422498  0.43463981]


In [30]:
# Väljastame atribuutide nimed kaalude järgi kahanevalt.
for f, c in sorted(zip(forest.feature_importances_, iris.feature_names), reverse=True):
    print(c, round(f, 3))

petal length (cm) 0.442
petal width (cm) 0.435
sepal length (cm) 0.098
sepal width (cm) 0.025


<a id='5_2'></a>
## Ülesanne 5.2

a) Viia DataFrame kujule varasematest ülesannetest tuttav UCI [loomaaia](http://archive.ics.uci.edu/ml/machine-learning-databases/zoo/) andmestik (või kasutada allpool toodud näitekoodi).

Millised kolmest atribuudist `aquatic`, `legs`, `type` on rohkem kui kahe võimaliku väärtusega? Milline nendest on nominaalatribuut (st selle väärtused pole sisuliselt  järjestatud)? Miks sellist atribuuti ei või käsitleda arvuna ja tuleks asendada $n$ kahendatribuudiga iga $n$  võimaliku väärtuse jaoks? Milline nendest kahest atribuudist  on arv ja milline informatsioon läheks masinõppe meetodite jaoks kaduma kui see asendada $n$ kahendatribuudiga? Kirjutada vastused ja põhjendus uude *Markdown* lahtrisse.

Konverteerida rohkem kui kahe väärtusega nominaalatribuut kahendkujule kasutades funktsiooni [pd.get_dummies(dataframe, columns=[col1, col2,..])](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html), kus `columns` on list veerunimedest, mida tuleb kahendkujule teisendada. Kuvada konverteeritud tabel.

Seejärel normaliseerida kogu tabel [0..1] skaalale. Kuvada veerg `legs`.


* Atribuudid `legs` ja `type` on rohkem kui kahe võimaliku väärtusega.
* Nendest `type` on nominaalatribuut, ning seda atribuuti ei saa käsitleda arvuna, sest selle väärtuseid ei saa sisuliselt järjestada
* `legs` väärtused on arvud ning kui see asendada n kahendatribuudiga, siis läheks kaduma informatsioon väärtuste järjestatuse kohta

In [31]:
# Andmete laadimine
import pandas as pd
zoo_df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/zoo/zoo.data", 
                     header=None, index_col=0,
                     names=["animal_name", "hair", "feathers", "eggs", "milk", "airborne", "aquatic", "predator", "toothed", "backbone", "breathes", "venomous", "fins", "legs", "tail", "domestic", "catsize", "type" ])
zoo_df

,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,type
animal_name,,,,,,,,,,,,,,,,,
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1


In [32]:
# Konverteerin nominaalatribuudi type kahendkujule
zoo_df = pd.get_dummies(zoo_df, columns=['type'])
zoo_df

,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,...,tail,domestic,catsize,type_1,type_2,type_3,type_4,type_5,type_6,type_7
animal_name,,,,,,,,,,,,,,,,,,,,,
aardvark,1,0,0,1,0,0,1,1,1,1,...,0,0,1,True,False,False,False,False,False,False
antelope,1,0,0,1,0,0,0,1,1,1,...,1,0,1,True,False,False,False,False,False,False
bass,0,0,1,0,0,1,1,1,1,0,...,1,0,0,False,False,False,True,False,False,False
bear,1,0,0,1,0,0,1,1,1,1,...,0,0,1,True,False,False,False,False,False,False
boar,1,0,0,1,0,0,1,1,1,1,...,1,0,1,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wallaby,1,0,0,1,0,0,0,1,1,1,...,1,0,1,True,False,False,False,False,False,False
wasp,1,0,1,0,1,0,0,0,0,1,...,0,0,0,False,False,False,False,False,True,False
wolf,1,0,0,1,0,0,1,1,1,1,...,1,0,1,True,False,False,False,False,False,False


In [33]:
# Normaliseerime andmed
norm_data = MinMaxScaler().fit_transform(zoo_df)
# Tekitame uue normaliseeritud df objekti
zoo_df_norm = pd.DataFrame(norm_data, columns=zoo_df.columns)
# Väljastan legs veeru
zoo_df_norm['legs']

0      0.50
1      0.50
2      0.00
3      0.50
4      0.50
       ... 
96     0.25
97     0.75
98     0.50
99     0.00
100    0.25
Name: legs, Length: 101, dtype: float64

b) Töödeldud andmestiku jaoks:
* Võtta  ennustatavaks klassiks jälle veerg `aquatic`, ja tõsta see klassivektorina ülejäänud andmetest välja (näiteks Dataframe [pop() meetodiga](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.pop.html)).
* Leida logistilise regressiooni mudel, kus nullist erinevate kaalude arv on L1 regulariseerimise abil minimeeritud, C=0.2. Millised atribuudid on logistilise regressiooni mudelis nullist erineva kaaluga (nimedega)?
* Hinnata atribuutide olulisust juhusliku metsa meetodil. Millised on olulised atribuudid?

In [34]:
# Tõsta veerg 'aquatic' ülejäänud andmetest välja
aquatic_column = zoo_df_norm.pop('aquatic')

In [35]:
# Leia logistilise regressiooni mudeli leitud kaalud, kui penalty='l1' ja C=0.1
log_regr_mudel = LogisticRegression(penalty="l1", C=0.2, solver='liblinear')
log_regr_mudel.fit(zoo_df_norm, aquatic_column)

log_regr_mudel.coef_ # atribuutide kaalud

array([[-0.58248928,  0.        ,  0.        ,  0.        ,  0.        ,
         0.37386116,  0.        ,  0.        , -0.9893942 ,  0.        ,
         0.9393786 ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ]])

In [36]:
# Millised atribuudid on nullist erineva kaaluga?

indeksid = np.nonzero(log_regr_mudel.coef_)[1] # leian indeksid, kus kaalud ei võrdu nulliga 
zoo_df_norm.columns[indeksid]


Index(['hair', 'predator', 'breathes', 'fins'], dtype='object')

In [37]:
# log_regr_mudel.predict(zoo_df_norm) # Annab ennustused, mis loomad elavad vees (aquatic = 1 või = 0) 


In [38]:
# # Ilma predict funktsiooni kasutamata tõenäosuse leidmine, et  nt loom indeksiga 3 elab vees
# zoo_df_norm.loc[[3]]
# logit_p = np.dot(log_regr_mudel.coef_, zoo_df_norm.iloc[3])
# # Sigmoidfunktsioon
# 1/(1+np.exp(-logit_p)) # annab tõenäosuse, et antud loom elab vees

In [41]:
# Hindand atribuutide olulisust juhusliku metsa meetodil
forest = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=-1)
forest.fit(zoo_df_norm, aquatic_column)
forest.feature_importances_ # annab atribuutide olulisused hindamaks, kas mingi loom on 'aquatic' - olulisuste summa on 1
print(pd.Series(forest.feature_importances_, index=zoo_df_norm.columns)) # annab kõrvuti antribuudi nime ja selle olulisuse
# Olulised atribuudid on: breathes, fins ja legs

NameError: name 'sort' is not defined

In [40]:
len(zoo_df.columns)

23